# 2. Merge everyone's data and train the model (one person)

Before running: every teammate's `part_*.csv` must be in the Drive folder below (including the final-test file with seeds 900000 and above).

Outputs: `models/ranker.joblib` (the trained model), `models/tree_rules.txt`, and `models/training_report.txt` (the results tables). They are copied to Drive and downloaded.

In [ ]:
# ---- EDIT THESE ----
REPO_URL        = "https://github.com/yugarora2/AI_Driven_CPU_SCHEDULING.git"
GIT_TAG         = "data-v1"       # must be the SAME code version the data was generated with
DRIVE_FOLDER    = "sched_data"
RUN_FINAL_TEST  = False           # set True ONCE, at the very end

## Set up

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/proj'):
    !git clone -q {REPO_URL} /content/proj
%cd /content/proj
if GIT_TAG:
    !git checkout -q {GIT_TAG}
!git log --oneline -1
!pip install -q -r requirements.txt

# No GitHub? Upload the project zip to Drive, then use these two lines instead of the clone above:
# !unzip -q /content/drive/MyDrive/adaptive_scheduler.zip -d /content/
# !mv /content/adaptive_scheduler /content/proj

## Copy the data files into the project and merge them

In [ ]:
import glob, shutil
os.makedirs("data", exist_ok=True)
files = sorted(glob.glob(f"/content/drive/MyDrive/{DRIVE_FOLDER}/part_*.csv"))
assert files, "No part_*.csv files found in your Drive folder"
for f in files:
    shutil.copy(f, "data/")
    print("copied", os.path.basename(f))
!python merge_data.py
!python inspect_data.py

## Train and evaluate

This can take a while with a lot of data (tens of minutes). It prints the results tables and saves the model.

In [ ]:
!mkdir -p models
!python train.py | tee models/training_report.txt

## Final test (run once, at the very end, and only after you are happy with everything else)

In [ ]:
if RUN_FINAL_TEST:
    get_ipython().system("python train.py --final | tee models/final_test_report.txt")
else:
    print("Skipped. Set RUN_FINAL_TEST = True above when you are ready (do it only once).")

## Save and download the model

Put `ranker.joblib` into the `models/` folder of your local project, then run the app.

In [ ]:
out = f"/content/drive/MyDrive/{DRIVE_FOLDER}/model_output"
os.makedirs(out, exist_ok=True)
!cp models/ranker.joblib models/tree_rules.txt models/training_report.txt {out}/
!cp models/final_test_report.txt {out}/ 2>/dev/null || true
print(f"ranker.joblib size: {os.path.getsize('models/ranker.joblib') / 1e6:.1f} MB")

from google.colab import files
files.download("models/ranker.joblib")
files.download("models/training_report.txt")
files.download("models/tree_rules.txt")